<a href="https://colab.research.google.com/github/tarun161/AILearning/blob/langchain_learning/LangChainLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-openai langchain-google-genai langchain-community langchain-core langchain-classic

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata

In [ ]:
# 1. Initialize different models
llm_openai = ChatOpenAI(model="gpt-4o", api_key=userdata.get("OPENAI_KEY"))
llm_gemini = ChatGoogleGenerativeAI(model="gemini-3.5-flash", api_key=userdata.get("GEMINI_API_KEY"))

prompt=ChatPromptTemplate.from_messages([
    ("system", "You are a witty {topic} expert. Explain concepts in a single sentence."),
    ("user", "Explain {concept}.")
  ])
print((prompt|llm_openai).invoke({"topic": "astrophysics", "concept": "black holes"}).content)
print((prompt|llm_gemini).invoke({"topic": "astrophysics", "concept": "black holes"}).content)

In [ ]:
# Formatting and output parsing
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class MovieReview(BaseModel):
    title: str = Field(description="Name of the movie in capital letters")
    rating: float = Field(description="Rating out of 10 should be Floating number not an Integer")
    summary: str = Field(description="One sentence summary")
parser=PydanticOutputParser(pydantic_object=MovieReview)

prompt=ChatPromptTemplate.from_messages([
    ("system","You are a movie critic.Analyze the movie.{format_instructions}"),
    ("user", "{movie_name}")
  ])

chain = prompt |llm_openai | parser
print(parser)
result = chain.invoke({
    "movie_name": "Attarintiki Daredi",
    "format_instructions": parser.get_format_instructions()
})
print(result)


In [ ]:
import os
from openai import OpenAI
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_PERSONAL')
openai_client = OpenAI()

# 2. Build your own robust image generation tool
@tool
def generate_image(prompt: str) -> str:
    """Generates an image based on a detailed text prompt. Returns the image URL."""
    try:
        # We use the official OpenAI SDK directly here
        response = openai_client.images.generate(
            model="gpt-image-1.5",
            prompt=prompt,
            size="1024x1024",
            n=1,
        )
        # Safely extract and return just the URL string
        print(response)
        return response.data[0].url
    except Exception as e:
        return f"Image generation failed: {str(e)}"

# 3. Add your custom tool to the list
tools = [generate_image]

# 4. Set up your LLM and Prompt
llm_openai = ChatOpenAI(model="gpt-4o", temperature=0.7, api_key=userdata.get("OPENAI_KEY"))

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative assistant. You can write text and generate images."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 5. Build and run the Agent
agent = create_tool_calling_agent(llm_openai, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True , max_iterations=1)

# Execute the test!
response = agent_executor.invoke({
    "input": "Write a 2-sentence story about a robotic cat on Mars, and then generate an image of it."
})

print("\nFinal Output:", response["output"])